# 04 · Pase y posesión 2v1

**Alcance:** experimento tabular reproducible sobre un modelo cinemático acotado. Las dinámicas son aproximaciones didácticas; las curvas que produzca este notebook pertenecen al modelo local. La conexión con `rcssserver` se revisa por separado cuando Docker esté disponible.

Consulta `../docs/passing_possession.md` para el MDP, las simplificaciones y los criterios de éxito.

## 1. Estado, acciones y recompensa

El estado discretiza posición del balón, vector al compañero, vector al defensor, pases y pasos de posesión (4×3×3×4×3×4×4×52 = 359 424 combinaciones; se almacena solo lo visitado). Las acciones son PASE, DRIBLE, GIRAR y DESPEJE. El defensor se aproxima al balón y puede cortar un pase. Se premia un pase completado (+30), se penaliza la intercepción (−30) y el balón fuera (−10). Éxito: conservar la posesión durante más de 50 pasos con ≥3 pases.

`reset()` inicia un episodio y `step(acción)` devuelve `(estado, recompensa, terminado, información)`. El estado contiene índices discretos; `información["success"]` identifica el criterio de éxito.

In [ ]:
import sys
from pathlib import Path

# Funciona con el kernel del contenedor y con un kernel local de VS Code.
for candidate in (
    Path("/workspace/src"),
    Path.cwd() / "src",
    Path.cwd().parent / "src",
):
    if candidate.is_dir():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError(
        "No se encontró src/. Abre el notebook desde el repositorio o notebooks/."
    )

In [ ]:
from metrics import summarize
from plotting import plot_training_comparison, plot_trajectory, plot_value_policy
from tabular import greedy_episode, train_q_learning
from tasks.passing_possession import PassingPossessionEnv

## 2. Entrenamiento y exploración

`train_q_learning()` actualiza Q tras cada transición. Se ejecutan dos experimentos independientes con la misma semilla: ε fijo = 0,2 y ε decreciente desde 1,0 (factor 0,995; mínimo 0,05). Solo cambia la estrategia de exploración.

In [ ]:
EPISODES = 2200
GAMMA = 0.99
ALPHA = 0.2
SEED = 42

experiments = {}
for label, mode, epsilon in (
    ("ε fijo", "constant", 0.2),
    ("ε decreciente", "decay", 1.0),
):
    env = PassingPossessionEnv(seed=SEED)
    q, history = train_q_learning(
        env,
        episodes=EPISODES,
        alpha=ALPHA,
        gamma=GAMMA,
        epsilon_mode=mode,
        epsilon_start=epsilon,
        seed=SEED,
    )
    experiments[label] = (q, history)

## 3. Retorno, éxito y pasos

`G₀` es el retorno descontado calculado desde el primer paso. La tasa de éxito es el porcentaje de episodios que alcanzan el criterio de la tarea. Las curvas usan un promedio móvil de 50 episodios; el resumen usa los últimos 100.

In [ ]:
for label, (_, history) in experiments.items():
    print(label, summarize(history))
plot_training_comparison(
    {label: history for label, (_, history) in experiments.items()}
)

## 4. Valor, política y trayectoria

El mapa representa V estimado = max Q y la acción greedy estimada. En estados con más de dos componentes, las otras dimensiones se fijan en la combinación más frecuente entre los estados visitados. El blanco significa «sin visitas». Después se ejecuta un episodio sin exploración para dibujar su trayectoria.

In [ ]:
q_decay = experiments["ε decreciente"][0]
plot_value_policy(
    q_decay, PassingPossessionEnv.state_shape, PassingPossessionEnv.action_names
)
evaluation_env = PassingPossessionEnv(seed=2026)
print(greedy_episode(evaluation_env, q_decay))
plot_trajectory(evaluation_env)

## 5. Conexión con RoboCup

Este notebook entrena en un entorno reducido. El cliente actual todavía no proporciona todas las observaciones necesarias para reproducir esta tarea dentro de `rcssserver` (por ejemplo, portero, postes, compañero o defensor según corresponda). La prueba de conexión UDP está en `01_ball_pursuit.ipynb`; los resultados de este notebook deben identificarse como simulados.